# iFood Case Técnico – Notebook 2: Modelagem

**Objetivo:** Desenvolver um modelo que, dado um par `(cliente, oferta)`, estime a probabilidade de conversão. Usar esse modelo para construir um sistema de recomendação personalizado e estimar o impacto no negócio.

**Abordagem:**
- **Problema:** Classificação binária — `label = 1` se a oferta foi convertida, `label = 0` caso contrário.
- **Modelo:** LightGBM com validação cruzada estratificada (5-fold).
- **Métricas principais:** AUC-ROC (discriminação) e Average Precision (considera desbalanceamento).
- **Recomendação:** Para cada cliente, recomendar a oferta que maximiza o **Net Expected Value** = P(conversão) × (valor médio do pedido − valor do desconto), ou seja, o lucro esperado por interação.

**Limitações conhecidas (temporal leakage):** As features comportamentais (`n_transactions`, `total_spent`, etc.) são computadas de todas as transações do período de teste, incluindo as que ocorrem *durante* o período da oferta. Em produção, essas features devem ser calculadas apenas com dados *anteriores* ao recebimento de cada oferta — isso elevaria artificialmente as features de quem converteu (label=1). O AUC reportado deve ser interpretado como um limite superior; o desempenho real seria ligeiramente menor.

## 1. Imports e Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.preprocessing import LabelEncoder
from itertools import product
import warnings, os, json

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.4f}".format

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")
RAW       = os.path.join(PROJECT_ROOT, "data", "raw")
PRESENTATION = os.path.join(PROJECT_ROOT, "presentation")
os.makedirs(PRESENTATION, exist_ok=True)
print("Ambiente configurado.")

## 2. Carregamento dos Dados Processados

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED, "interactions_features.csv"))
offers_raw = pd.read_json(os.path.join(RAW, "offers.json"))

print(f"Shape: {df.shape}")
print(f"\nColunas: {list(df.columns)}")
print(f"\nDistribuição do target:")
print(df["label"].value_counts())
print(f"\nTaxa de conversão: {df['label'].mean():.2%}")
df.head(3)

## 3. Análise Exploratória (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Taxa de conversão por tipo de oferta
conv_by_type = df.groupby("offer_type")["label"].mean().sort_values(ascending=False)
conv_by_type.plot(kind="bar", ax=axes[0], color=["#e74c3c","#3498db","#2ecc71"])
axes[0].set_title("Conversão por Tipo de Oferta")
axes[0].set_ylabel("Taxa de Conversão")
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].tick_params(axis="x", rotation=0)
for bar, val in zip(axes[0].patches, conv_by_type):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.1%}", ha="center", va="bottom", fontsize=9)

# Volume de interações por tipo
vol_by_type = df.groupby("offer_type")["label"].count()
vol_by_type.plot(kind="bar", ax=axes[1], color=["#95a5a6","#7f8c8d","#bdc3c7"])
axes[1].set_title("Volume de Interações por Tipo")
axes[1].set_ylabel("Interações")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

# Conversão por gênero
conv_by_gender = df.groupby("gender")["label"].mean().sort_values(ascending=False)
conv_by_gender.plot(kind="bar", ax=axes[2], color=["#9b59b6","#e67e22","#1abc9c","#34495e"])
axes[2].set_title("Conversão por Gênero")
axes[2].set_ylabel("Taxa de Conversão")
axes[2].set_xlabel("")
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[2].tick_params(axis="x", rotation=0)

plt.suptitle("EDA – Taxas de Conversão", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "eda_conversion_rates.png"), bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de idade por label
for label, color, name in [(0, "#e74c3c", "Não Converteu"), (1, "#2ecc71", "Converteu")]:
    subset = df[df["label"] == label]["age"].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title("Distribuição de Idade por Conversão")
axes[0].set_xlabel("Idade")
axes[0].set_ylabel("Densidade")
axes[0].legend()

# Ticket médio por taxa de conversão (scatter por oferta)
offer_stats = df.groupby("offer_id").agg(
    conversion_rate=("label", "mean"),
    avg_discount=("discount_value", "first"),
    min_value=("min_value", "first"),
    offer_type=("offer_type", "first"),
    n=("label", "count")
).reset_index()

colors = {"bogo": "#e74c3c", "discount": "#3498db", "informational": "#2ecc71"}
for otype, group in offer_stats.groupby("offer_type"):
    axes[1].scatter(group["min_value"], group["conversion_rate"],
                    s=group["n"]/5, color=colors[otype], label=otype, alpha=0.8)
axes[1].set_title("Conversão × Valor Mínimo (tamanho = volume)")
axes[1].set_xlabel("Valor Mínimo da Oferta")
axes[1].set_ylabel("Taxa de Conversão")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].legend(title="Tipo")

plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "eda_demographics.png"), bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de gasto total por label
for label, color, name in [(0, "#e74c3c", "Não Converteu"), (1, "#2ecc71", "Converteu")]:
    subset = df[df["label"] == label]["total_spent"].dropna()
    axes[0].hist(np.log1p(subset), bins=40, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title("Distribuição de Gasto Total (log)")
axes[0].set_xlabel("log(1 + Total Gasto)")
axes[0].set_ylabel("Densidade")
axes[0].legend()

# Heatmap: conversão média por (offer_type × canal)
channel_cols = ["has_email", "has_mobile", "has_social", "has_web"]
pivot_data = []
for ch in channel_cols:
    for otype in df["offer_type"].unique():
        mask = (df["offer_type"] == otype) & (df[ch] == 1)
        rate = df.loc[mask, "label"].mean()
        pivot_data.append({"channel": ch.replace("has_",""), "offer_type": otype, "rate": rate})
pivot_df = pd.DataFrame(pivot_data).pivot(index="channel", columns="offer_type", values="rate")
sns.heatmap(pivot_df, annot=True, fmt=".2%", cmap="YlOrRd", ax=axes[1], linewidths=0.5)
axes[1].set_title("Conversão Média: Canal × Tipo de Oferta")
axes[1].set_xlabel("Tipo de Oferta")
axes[1].set_ylabel("Canal")

plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "eda_behavior.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Correlação das features numéricas com o label
numeric_cols = [
    "age", "credit_card_limit", "account_age_days",
    "min_value", "duration", "discount_value", "n_channels",
    "n_transactions", "total_spent", "avg_transaction", "avg_daily_spend",
    "discount_to_limit_ratio", "min_value_vs_avg_spend"
]
corr_with_label = df[numeric_cols + ["label"]].corr()["label"].drop("label").sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
corr_with_label.plot(kind="barh", ax=ax,
    color=["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_label])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlação das Features com Conversão (label)")
ax.set_xlabel("Correlação de Pearson")
plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "eda_correlation.png"), bbox_inches="tight")
plt.show()

## 4. Preparação das Features

In [ ]:
FEATURE_COLS = [
    # Perfil do cliente
    "age", "credit_card_limit", "account_age_days", "is_complete_profile",
    # Oferta
    "min_value", "duration", "discount_value", "n_channels",
    "has_email", "has_mobile", "has_social", "has_web",
    # Comportamento transacional
    "n_transactions", "total_spent", "avg_transaction", "std_transaction",
    "observation_window", "avg_daily_spend",
    # Features de interação cliente×oferta
    "discount_to_limit_ratio", "min_value_vs_avg_spend",
    # Categóricas (codificadas abaixo)
    "offer_type_enc", "gender_enc",
]

df_model = df.copy()

# Encoding de categóricas
le_offer = LabelEncoder()
le_gender = LabelEncoder()
df_model["offer_type_enc"] = le_offer.fit_transform(df_model["offer_type"].fillna("Unknown"))
df_model["gender_enc"]     = le_gender.fit_transform(df_model["gender"].fillna("Unknown"))

X = df_model[FEATURE_COLS].copy()
y = df_model["label"]

print(f"Features de treino: {X.shape[1]}")
print(f"Amostras: {X.shape[0]:,}")
print(f"\nBalanceamento das classes:")
print(y.value_counts())
print(f"Proporção positivos: {y.mean():.2%}")

## 5. Treinamento do Modelo – LightGBM com Validação Cruzada

**Por que LightGBM?**
- Excelente performance em dados tabulares com features mistas (numéricas + categóricas).
- Lida nativamente com valores nulos (sem imputação obrigatória).
- `is_unbalance=True` ajusta automaticamente os pesos para classes desbalanceadas.
- Treinamento rápido com `early_stopping` para evitar overfitting.

In [ ]:
params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 20,
    "colsample_bytree": 0.8,
    "subsample": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "is_unbalance": True,
    "random_state": 42,
    "verbose": -1,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models = []
val_aucs = []
val_aps = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ]
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_ap  = average_precision_score(y_val, oof_preds[val_idx])
    val_aucs.append(fold_auc)
    val_aps.append(fold_ap)
    models.append(model)
    print(f"Fold {fold+1} | best_iter={model.best_iteration_:>4} | AUC={fold_auc:.4f} | AP={fold_ap:.4f}")

print(f"\n{'='*55}")
print(f"OOF AUC: {np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}")
print(f"OOF AP:  {np.mean(val_aps):.4f} ± {np.std(val_aps):.4f}")

## 6. Avaliação do Modelo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC (out-of-fold)
fpr, tpr, _ = roc_curve(y, oof_preds)
auc_score = roc_auc_score(y, oof_preds)
axes[0].plot(fpr, tpr, color="#e74c3c", lw=2, label=f"AUC = {auc_score:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="#e74c3c")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Curva ROC (Out-of-Fold)")
axes[0].legend(loc="lower right")

# Matriz de confusão (threshold = 0.5)
threshold = 0.5
y_pred = (oof_preds >= threshold).astype(int)
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["Não Converteu", "Converteu"],
            yticklabels=["Não Converteu", "Converteu"])
axes[1].set_title(f"Matriz de Confusão (threshold={threshold})")
axes[1].set_xlabel("Predito")
axes[1].set_ylabel("Real")

plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "model_evaluation.png"), bbox_inches="tight")
plt.show()

print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=["Não Converteu", "Converteu"]))

In [ ]:
# Análise de threshold: precision, recall e F1 vs threshold
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds_pr = precision_recall_curve(y, oof_preds)
f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
best_threshold_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds_pr[best_threshold_idx]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds_pr, precision[:-1], label="Precision", color="#3498db")
ax.plot(thresholds_pr, recall[:-1], label="Recall", color="#e74c3c")
ax.plot(thresholds_pr, f1_scores[:-1], label="F1", color="#2ecc71", lw=2)
ax.axvline(best_threshold, color="black", linestyle="--", alpha=0.6,
           label=f"Melhor F1 threshold = {best_threshold:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision, Recall e F1 por Threshold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Melhor threshold para F1: {best_threshold:.2f}")
print(f"F1 no melhor threshold: {f1_scores[best_threshold_idx]:.4f}")
print(f"Precision: {precision[best_threshold_idx]:.4f}")
print(f"Recall:    {recall[best_threshold_idx]:.4f}")

## 7. Importância das Features

In [ ]:
# Média da importância de gain entre todos os folds
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": np.mean([m.booster_.feature_importance(importance_type="gain") for m in models], axis=0)
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(importance_df["feature"][::-1], importance_df["importance"][::-1],
               color=plt.cm.viridis(np.linspace(0.2, 0.9, len(importance_df))))
ax.set_title("Importância das Features (Gain – média 5 folds)")
ax.set_xlabel("Importância (Gain)")
plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "feature_importance.png"), bbox_inches="tight")
plt.show()

print("Top 10 features:")
print(importance_df.head(10).to_string(index=False))

## 7.1 Discussão: Temporal Leakage e AUC

O AUC de ~0.92 é alto. A análise de importância ajuda a diagnosticar se isso se deve a leakage temporal ou a sinal genuíno:

- **Se `total_spent` / `n_transactions` dominam**: sinal de leakage — quem converteu fez mais compras *durante* a oferta, inflando suas features.
- **Se `min_value_vs_avg_spend` / `discount_value` / `offer_type_enc` dominam**: sinal genuíno — o modelo aprendeu compatibilidade oferta×perfil.

**Próximos passos para mitigar o leakage em produção:**
1. Joinar cada interação com as features do cliente calculadas apenas com `time < received_at`.
2. Usar `time_since_test_start` como ponto de corte ao agregar transações no notebook 1.

## 8. Sistema de Recomendação Personalizado

Para cada cliente, geramos probabilidades de conversão para todas as 10 ofertas. A oferta recomendada é a que maximiza o **Net Expected Value**:

```
Net EV = P(conversão) × (valor_médio_pedido − desconto)
```

Essa métrica maximiza o *lucro* esperado por oferta enviada, não apenas a taxa de conversão. Ofertas com descontos elevados (BOGO) são preferidas somente quando a probabilidade de conversão compensa o custo maior. Ofertas *informational* têm desconto=0, portanto Net EV = 0 e nunca são recomendadas (elas têm outro papel: awareness/brand).

> **Premissa:** `valor_médio_pedido` é estimado pela mediana do ticket médio histórico dos clientes no dataset.

In [ ]:
# Valor médio de pedido estimado pela mediana do ticket médio dos clientes
avg_order_value = df["avg_transaction"].median()
print(f"Ticket médio estimado: R$ {avg_order_value:.2f}")

# Extrair features únicas por cliente (usando a última ocorrência)
client_features = [
    "account_id", "age", "credit_card_limit", "account_age_days", "is_complete_profile",
    "n_transactions", "total_spent", "avg_transaction", "std_transaction",
    "observation_window", "avg_daily_spend", "gender"
]
customer_df = (
    df_model
    .groupby("account_id")
    .last()
    .reset_index()
    [client_features]
)

# Features únicas por oferta (apenas não-informacionais para recomendação com Net EV)
offer_features = [
    "offer_id", "offer_type", "min_value", "duration", "discount_value",
    "n_channels", "has_email", "has_mobile", "has_social", "has_web"
]
offer_df = df.drop_duplicates("offer_id")[offer_features]

# Cross join: todas combinações cliente × oferta
all_pairs = customer_df.merge(offer_df, how="cross")

# Features derivadas
all_pairs["discount_to_limit_ratio"] = np.where(
    all_pairs["credit_card_limit"] > 0,
    all_pairs["discount_value"] / all_pairs["credit_card_limit"],
    0
)
all_pairs["min_value_vs_avg_spend"] = np.where(
    all_pairs["avg_transaction"] > 0,
    all_pairs["min_value"] / all_pairs["avg_transaction"],
    np.nan
)

# Encoding
all_pairs["offer_type_enc"] = le_offer.transform(
    all_pairs["offer_type"].fillna("Unknown")
)
all_pairs["gender_enc"] = le_gender.transform(
    all_pairs["gender"].fillna("Unknown")
)

# Scorer: média do ensemble de 5 modelos
X_all = all_pairs[FEATURE_COLS].fillna(0)
all_preds = np.mean(
    [m.predict_proba(X_all)[:, 1] for m in models],
    axis=0
)
all_pairs["p_conversion"] = all_preds

# Net EV = P(conversão) × (ticket médio − desconto)
# Para informational: discount=0 → net_margin=avg_order → mas label=viewed (awareness)
# Recomendação foca em ofertas com real call-to-action (bogo/discount)
all_pairs["net_margin"] = avg_order_value - all_pairs["discount_value"]
all_pairs["net_ev"] = all_pairs["p_conversion"] * all_pairs["net_margin"].clip(lower=0)

# Manter expected_value original também para comparação
all_pairs["expected_value"] = all_pairs["p_conversion"] * all_pairs["discount_value"]

print(f"Pares avaliados: {len(all_pairs):,}")
print(f"Clientes únicos: {all_pairs['account_id'].nunique():,}")
print(f"Ofertas únicas:  {all_pairs['offer_id'].nunique():,}")
print(f"\nNet EV médio por tipo de oferta:")
print(all_pairs.groupby("offer_type")[["p_conversion","net_margin","net_ev"]].mean().round(3))

In [ ]:
# Recomendação: oferta com maior Net EV por cliente
# Exclui informational (não geram pedidos diretos)
non_info_pairs = all_pairs[all_pairs["offer_type"] != "informational"]

recommendations = (
    non_info_pairs
    .sort_values(["account_id", "net_ev"], ascending=[True, False])
    .groupby("account_id")
    .first()
    .reset_index()
    [["account_id", "offer_id", "offer_type", "p_conversion", "net_ev",
      "discount_value", "min_value", "duration"]]
)

print(f"Recomendações geradas: {len(recommendations):,} clientes")
print(f"\nDistribuição das ofertas recomendadas (por Net EV):")
print(recommendations.groupby("offer_type")[["p_conversion","net_ev"]].agg(["count","mean"]).round(3))

print(f"\nP(conversão) médio nas recomendações: {recommendations['p_conversion'].mean():.2%}")
print(f"Net EV médio:                          {recommendations['net_ev'].mean():.2f}")
recommendations.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição das ofertas recomendadas
rec_by_type = recommendations["offer_type"].value_counts()
rec_by_type.plot(kind="bar", ax=axes[0], color=["#e74c3c","#3498db"])
axes[0].set_title("Distribuição das Ofertas Recomendadas (Net EV)")
axes[0].set_ylabel("Nº de Clientes")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
for bar, val in zip(axes[0].patches, rec_by_type):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"{val:,}\n({val/len(recommendations):.1%})",
                 ha="center", va="bottom", fontsize=8)

# Distribuição do Net EV predito
axes[1].hist(recommendations["net_ev"], bins=50, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].axvline(recommendations["net_ev"].mean(), color="red", linestyle="--",
               label=f"Média: R$ {recommendations['net_ev'].mean():.2f}")
axes[1].set_title("Distribuição de Net EV por Cliente (Recomendações)")
axes[1].set_xlabel("Net Expected Value (R$)")
axes[1].set_ylabel("Frequência")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "recommendation_distribution.png"), bbox_inches="tight")
plt.show()

## 9. Impacto de Negócio

### Metodologia
Comparamos três estratégias de distribuição de ofertas (excluindo informational, que não gera pedidos diretos):

1. **Baseline Random**: Sortear oferta aleatoriamente entre bogo/discount.
2. **Baseline Top-1 Fixo**: Enviar sempre a oferta com maior conversão histórica global.
3. **Modelo Personalizado (Net EV)**: Enviar a oferta que maximiza P(conv) × (ticket_médio − desconto).

**Premissas do cálculo:**
- Cada conversão gera 1 pedido com valor = `avg_transaction` (mediana histórica do cliente).
- Custo da oferta = desconto pago ao cliente por conversão.
- Resultado Líquido = Receita − Custo.

> **Nota:** Este é um modelo simplificado de impacto. Em produção, seria necessário um teste A/B controlado para estimar o incremento real (lift causal), separando o efeito da oferta do comportamento orgânico do cliente.

In [ ]:
# --- Baseline 1: Oferta aleatória (não-informacional) ---
non_info_df = df[df["offer_type"] != "informational"]
baseline_random_rate    = non_info_df["label"].mean()
baseline_random_discount = non_info_df["discount_value"].mean()

# --- Baseline 2: Melhor oferta global (maior conversão histórica) ---
best_offer_id = non_info_df.groupby("offer_id")["label"].mean().idxmax()
best_offer_rate     = non_info_df[non_info_df["offer_id"] == best_offer_id]["label"].mean()
best_offer_discount = df[df["offer_id"] == best_offer_id]["discount_value"].iloc[0]
print(f"Melhor oferta fixa: {best_offer_id[:8]}… "
      f"| Conv={best_offer_rate:.2%} | Desconto=R${best_offer_discount}")

# --- Modelo Personalizado ---
# A taxa de conversão estimada = média ponderada das P(conversão) recomendadas
model_rate           = recommendations["p_conversion"].mean()
model_avg_discount   = recommendations["discount_value"].mean()

# --- Cálculo de Resultado Líquido ---
n_customers = len(recommendations)

strategies = {
    "Random (bogo/discount)": {
        "rate": baseline_random_rate,
        "avg_discount": baseline_random_discount,
    },
    "Melhor Oferta Fixa": {
        "rate": best_offer_rate,
        "avg_discount": best_offer_discount,
    },
    "Modelo Personalizado": {
        "rate": model_rate,
        "avg_discount": model_avg_discount,
    },
}

results = []
for name, s in strategies.items():
    conversions = n_customers * s["rate"]
    revenue = conversions * avg_order_value
    cost    = conversions * s["avg_discount"]
    net     = revenue - cost
    results.append({
        "Estratégia":              name,
        "Taxa Conversão":          s["rate"],
        "Conversões Estimadas":    int(conversions),
        "Desconto Médio (R$)":     s["avg_discount"],
        "Receita Gerada (R$)":     revenue,
        "Custo Desconto (R$)":     cost,
        "Resultado Líquido (R$)":  net,
    })

results_df = pd.DataFrame(results)
results_num = results_df.copy()  # keep numeric for chart

results_df["Taxa Conversão"] = results_df["Taxa Conversão"].map("{:.2%}".format)
results_df["Desconto Médio (R$)"] = results_df["Desconto Médio (R$)"].map("R$ {:.2f}".format)
for col in ["Receita Gerada (R$)", "Custo Desconto (R$)", "Resultado Líquido (R$)"]:
    results_df[col] = results_df[col].map("R$ {:,.0f}".format)

print(results_df[["Estratégia","Taxa Conversão","Desconto Médio (R$)",
                   "Resultado Líquido (R$)"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#95a5a6", "#3498db", "#e74c3c"]
labels = [r["Estratégia"] for r in results]
rates  = [r["Taxa Conversão"] for r in results]
nets   = [r["Resultado Líquido (R$)"] for r in results]

# Taxas de conversão
bars = axes[0].bar(labels, rates, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_title("Taxa de Conversão Estimada por Estratégia")
axes[0].set_ylabel("Taxa de Conversão")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for bar, val in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{val:.2%}", ha="center", va="bottom", fontweight="bold")
axes[0].tick_params(axis="x", rotation=15)

# Resultado líquido
bars2 = axes[1].bar(labels, nets, color=colors, edgecolor="white", linewidth=1.5)
axes[1].set_title("Resultado Líquido Estimado por Estratégia")
axes[1].set_ylabel("Resultado Líquido (R$)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
for bar, val in zip(bars2, nets):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f"R$ {val:,.0f}", ha="center", va="bottom", fontweight="bold", fontsize=8)
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Impacto de Negócio: Distribuição Personalizada vs Baselines",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PRESENTATION, "business_impact.png"), bbox_inches="tight")
plt.show()

# Lift do modelo
baseline_net = results[0]["Resultado Líquido (R$)"]
model_net    = results[2]["Resultado Líquido (R$)"]
print(f"\nLift em taxa de conversão (modelo vs random): {rates[2]/rates[0]:.2f}x")
print(f"Lift em resultado líquido  (modelo vs random): {model_net/baseline_net:.2f}x")
print(f"Incremento absoluto em resultado:              R$ {model_net - baseline_net:,.0f}")

In [ ]:
# Análise por segmento de valor do cliente
rec_full = recommendations.merge(
    df[["account_id", "account_age_days", "n_transactions", "total_spent"]]
    .drop_duplicates("account_id"),
    on="account_id", how="left"
)

rec_full["segment"] = pd.qcut(
    rec_full["total_spent"].fillna(0),
    q=4,
    labels=["Low Value", "Mid Value", "High Value", "Top Value"]
)

print("\nP(conversão) e Net EV médios por segmento de valor do cliente:")
seg_analysis = rec_full.groupby("segment", observed=True).agg(
    n_clientes=("account_id", "count"),
    p_conv_media=("p_conversion", "mean"),
    net_ev_medio=("net_ev", "mean"),
    desconto_medio=("discount_value", "mean"),
).round(3)
print(seg_analysis)

## 10. Salvando Resultados

In [ ]:
# Salva tabela de scores completa e recomendações finais
rec_export = all_pairs.sort_values(["account_id", "net_ev"], ascending=[True, False])
rec_export.to_csv(os.path.join(PROCESSED, "all_offer_scores.csv"), index=False)
recommendations.to_csv(os.path.join(PROCESSED, "recommendations.csv"), index=False)

print(f"Scores de todas as combinações: {len(rec_export):,} linhas")
print(f"Arquivo: {os.path.join(PROCESSED, 'all_offer_scores.csv')}")
print(f"\nRecomendações finais: {len(recommendations):,} linhas")
print(f"Arquivo: {os.path.join(PROCESSED, 'recommendations.csv')}")
print("\nPipeline de modelagem concluído!")